In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

#datasets' path
DATA_DIR = Path("../data")
GBD_PATH = DATA_DIR / "raw/GBD_raw.csv"
WDI_PATH = DATA_DIR / "raw/WDI_raw.csv"

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

#read the dataset
gbd = pd.read_csv(GBD_PATH)
wdi = pd.read_csv(WDI_PATH)
print("GBD shape:", gbd.shape)
print("WDI shape:", wdi.shape)

GBD shape: (110, 11)
WDI shape: (49, 14)


In [3]:
#instead of modifying the original datasets, doing it on their copy is safer
gbd_clean = gbd.copy()
wdi_clean = wdi.copy()

In [ ]:
#in the WDI dataset, there're some lines/footer that is not data to analyze
#remove them. only keep rows with valid country and indicators
wdi_clean = wdi_clean[
    wdi_clean["Country Code"].notna() &
    wdi_clean["Series Code"].notna()
].copy()
print("WDI shape after cleaning:", wdi_clean.shape)

WDI shape after cleaning: (44, 14)


In [7]:
#Since the WDI data's each row is a country's each indicators while its columns is the corresponding number of each year
#we need it to transform from wide format _> long format (each line is a country's indicator in a year)
year_columns = [
    col for col in wdi_clean.columns
    if "[YR" in col
]
wdi_long = wdi_clean.melt(
    id_vars=[
        "Country Name",
        "Country Code",
        "Series Name",
        "Series Code"
    ],
    value_vars=year_columns,
    var_name="year",
    value_name="value"
)
wdi_long["year"] = (
    wdi_long["year"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

#now transform from long -> wide (indicators into 4 separated columns)
wdi_wide = wdi_long.pivot(
    index=["Country Code", "Country Name", "year"],
    columns="Series Code",
    values="value"
).reset_index()
wdi_wide.columns.name = None
#and rename them for convenience
wdi_wide = wdi_wide.rename(columns={
    "Country Name": "country",
    "year": "year",
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "SH.XPD.OOPC.CH.ZS": "oop_expenditure",
    "SH.XPD.CHEX.PC.CD": "health_expenditure_per_capita",
    "SP.URB.TOTL.IN.ZS": "urban_population"
})

In [8]:
#the two dataset are not using the identical country names
country_mapping = {
    "Lao PDR": "Lao People's Democratic Republic"
}
wdi_wide["country"] = (
    wdi_wide["country"].replace(country_mapping)
)
gbd_clean["country"] = gbd_clean["location_name"]

#check
gbd_countries = set(gbd_clean["country"].unique())
wdi_countries = set(wdi_wide["country"].unique())
print("Countries only in GBD:")
print(sorted(gbd_countries - wdi_countries))
print("Countries only in WDI:")
print(sorted(wdi_countries - gbd_countries))

Countries only in GBD:
[]
Countries only in WDI:
[]


In [15]:
#check for the number of countries, years and observations
#GBD
print("---GBD Dataset---")
print("Number of countries:", gbd_clean["location_name"].nunique())
print("Number of years:", gbd_clean["year"].nunique())
print("Number of observations:", len(gbd_clean))
print("\nYears available:")
print(sorted(gbd_clean["year"].unique()))
print("Countries available:")
print(sorted(gbd_clean["location_name"].unique()))

#WDI
print("\n---WDI Dataset---")
print("Number of countries:", wdi_wide["country"].nunique())
print("Number of years:", wdi_wide["year"].nunique())
print("Number of observations:", len(wdi_wide))
print("\nYears available:")
print(sorted(wdi_wide["year"].unique()))
print("Countries available:")
print(sorted(wdi_wide["country"].unique()))

#check number of observations by each contry, to see whether a country misses a year
print("\nGBD observations per country:")
display(
    gbd_clean.groupby("location_name")["year"]
    .nunique()
    .sort_values()
)
print("WDI observations per country:")
display(
    wdi_wide.groupby("country")["year"]
    .nunique()
    .sort_values()
)

---GBD Dataset---
Number of countries: 11
Number of years: 10
Number of observations: 110

Years available:
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Countries available:
['Brunei Darussalam', 'Cambodia', 'Indonesia', "Lao People's Democratic Republic", 'Malaysia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Timor-Leste', 'Viet Nam']

---WDI Dataset---
Number of countries: 11
Number of years: 10
Number of observations: 110

Years available:
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Countries available:
['Brunei Darussalam', 'Cambodia', 'Indonesia', "Lao People's Democratic Republic", 'Malaysia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Timor-Leste', 'Viet Nam']

GBD observations per country:


location_name
Brunei Darussalam                   10
Cambodia                            10
Indonesia                           10
Lao People's Democratic Republic    10
Malaysia                            10
Myanmar                             10
Philippines                         10
Singapore                           10
Thailand                            10
Timor-Leste                         10
Viet Nam                            10
Name: year, dtype: int64

WDI observations per country:


country
Brunei Darussalam                   10
Cambodia                            10
Indonesia                           10
Lao People's Democratic Republic    10
Malaysia                            10
Myanmar                             10
Philippines                         10
Singapore                           10
Thailand                            10
Timor-Leste                         10
Viet Nam                            10
Name: year, dtype: int64

In [24]:
#merge the two dataset
#choose target variables from GBD
gbd_final = gbd_clean[
    ["location_name", "year", "val", "lower", "upper"] #there're two other column Upper and Lower in the dataset, we'll keep them in the raw data for certainty analysis
].copy()
gbd_final = gbd_final.rename(columns={
    "location_name": "country",
    "val": "daly_rate", #just change the name to be clearer
    "lower": "daly_lower",
    "upper": "daly_upper"
})
#then merge by two keys: country and year
merged = pd.merge(
    gbd_final,
    wdi_wide,
    on=["country", "year"],
    how="inner",
    validate="one_to_one" #to detect if a countr-year pair is duplicated in 2 dts
)

In [25]:
#validation
print("Merged shape:", merged.shape)
print("Countries:", merged["country"].nunique())
print("Years:", sorted(merged["year"].unique()))
print("Missing values:")
display(merged.isnull().sum())
print(
    "Duplicate Country-Year:",
    merged.duplicated(["country", "year"]).sum()
)

assert merged.duplicated(
    ["country", "year"]
).sum() == 0
assert merged["daly_rate"].notna().all()
assert merged["country"].nunique() == 11


Merged shape: (110, 10)
Countries: 11
Years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Missing values:


country                          0
year                             0
daly_rate                        0
daly_lower                       0
daly_upper                       0
Country Code                     0
gdp_per_capita                   0
health_expenditure_per_capita    0
oop_expenditure                  0
urban_population                 0
dtype: int64

Duplicate Country-Year: 0


In [26]:
#export into one dataset
final_columns = [
    "country",
    "year",
    "daly_rate",
    "daly_lower",
    "daly_upper",
    "gdp_per_capita",
    "oop_expenditure",
    "health_expenditure_per_capita",
    "urban_population"
]

merged = merged[final_columns].sort_values(
    ["country", "year"]
).reset_index(drop=True)
output_path = PROCESSED_DIR / "merged_dataset.csv"
merged.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)

Dataset saved to: ..\data\processed\merged_dataset.csv
